# SOLUTION: Edge Cases & Questionable Applicability of Statistical Tests
## Technical Depth on When and Why Chi-Square, t-test, and ANOVA Break Down


## Extended Decision Framework
The original flowchart provides a useful first-order decision rule. However, real data frequently exists in the "gray zones" where the classical procedures are asymptotically justified but perform poorly in finite samples. The extended framework below incorporates robustness considerations.


## 1. Edge Cases for the Chi-Square Test of Independence

### Scenario A: Small Expected Cell Counts
The chi-square approximation to the sampling distribution of the Pearson statistic relies on the central limit theorem applied to cell frequencies. When expected cell counts are small (commonly cited rule: >20% of cells have expected count < 5, or any cell < 1), the approximation deteriorates. The actual Type I error rate can deviate substantially from the nominal α.

**Technical consequence**: The test becomes either conservative or anti-conservative depending on the pattern of sparseness. The likelihood ratio statistic (G²) often behaves better than Pearson’s X² in sparse tables.

**Recommended approach**:
- For 2×2 tables: Use **Fisher’s Exact Test** (conditions on the marginal totals and enumerates the hypergeometric distribution).
- For larger tables with moderate sparseness: Use **exact tests via Monte Carlo simulation** or the **likelihood ratio test** with caution.
- Modern recommendation: Many methodologists now prefer **Fisher’s Exact** or **Barnard’s test** even for moderately sparse 2×2 tables.

### Scenario B: Dependent / Paired Observations
The chi-square test assumes that the n observations are independent. When the same individuals are measured at two time points (or under two conditions), the observations are paired. The off-diagonal cells in the contingency table are no longer independent.

**Technical consequence**: The variance of the test statistic is underestimated, leading to inflated Type I error rates. The dependence structure violates the multinomial sampling model underlying the chi-square distribution.

**Correct procedure**: Use **McNemar’s test** (for 2×2) or its generalization (Stuart-Maxwell test) for larger tables. These tests focus on the discordant pairs and properly account for the dependence.

### Scenario C: Non-Mutually Exclusive Categories
Chi-square requires that each observation falls into exactly one cell of the contingency table. When individuals can belong to multiple categories simultaneously (e.g., multiple social media platforms), the data-generating process is no longer multinomial.

**Technical consequence**: The row and column totals no longer have a clear probabilistic interpretation, and the chi-square statistic does not follow its nominal distribution.

**Recommended approach**:
- Recode into mutually exclusive categories if theoretically justified.
- Model each platform as a separate binary outcome and use logistic regression (or multivariate approaches).
- For multiple-response data, specialized techniques such as multiple correspondence analysis or item response models are more appropriate.


## 2. Edge Cases for the Two-Sample t-test

### Scenario D: Extreme Skewness + Very Small Sample Size
The t-test relies on the sampling distribution of the difference in means being approximately normal (via the Central Limit Theorem) and on the sample variance being a good estimator of the population variance.

With n ≈ 8–10 and strong skewness (e.g., exponential or lognormal data), the sampling distribution of the mean remains noticeably skewed. The t-statistic therefore does not follow a t-distribution closely enough, especially in the tails.

**Technical consequence**:
- Inflated Type I error in one tail and conservative behavior in the other.
- Confidence intervals have poor coverage.
- Power can be substantially lower than nominal.

**Robust alternatives**:
- **Yuen’s t-test** (trimmed means + Winsorized variance) — trims a fixed percentage (commonly 20%) from each tail before computing location and scale.
- **Bootstrap-t** or **percentile bootstrap** methods.
- Non-parametric tests (Mann-Whitney) when the research question can be reframed in terms of stochastic dominance.

### Scenario E: Severe Heteroscedasticity + Small n
When the ratio of variances exceeds ~3–4 and sample sizes are small and/or unequal, even the Welch t-test (which adjusts degrees of freedom) can have suboptimal performance because the Welch-Satterthwaite approximation itself becomes less accurate.

**Technical consequence**: The actual Type I error rate can exceed the nominal level, and power is reduced.

**Better options**:
- Yuen’s t-test with trimmed means is often more robust to both non-normality and heteroscedasticity.
- Bootstrap methods that do not rely on the t-distribution.

### Scenario F: Influence of Outliers
A single extreme observation can arbitrarily inflate both the mean and the standard deviation, moving the t-statistic in unpredictable directions.

**Technical consequence**: The breakdown point of the classical mean and variance is 1/n. One contaminated observation can completely dominate the result.

**Robust approach**: Use estimators with high breakdown points (e.g., trimmed means, M-estimators, or median-based tests).


## 3. Edge Cases for ANOVA and Tukey’s HSD

### Scenario G: Heteroscedasticity + Unequal Group Sizes
The classical ANOVA F-test assumes homogeneity of variance (homoscedasticity). When this assumption is violated and group sizes differ, the F-statistic is no longer distributed as central F under the null, and the actual Type I error rate can be seriously distorted.

The distortion is worst when larger variances are paired with smaller sample sizes (positive pairing) or vice versa.

**Technical consequence**: The F-test can be either liberal or conservative. Tukey’s HSD, which uses the pooled within-group variance (MSE), inherits the same problem and produces confidence intervals with incorrect coverage.

**Recommended procedures**:
- **Welch’s ANOVA** (one-way) + **Games-Howell** post-hoc test (does not assume equal variances).
- Robust ANOVA methods based on trimmed means (e.g., from the `WRS2` package in R).

### Scenario H: Non-Normality in Small Groups
ANOVA is known to be relatively robust to moderate non-normality when group sizes are equal and reasonably large (n > 20–30). However, with very small groups (n < 10) and clear departures from normality (bimodality, heavy tails), both the F-test and the subsequent Tukey comparisons lose their nominal properties.

**Technical consequence**: The F-distribution approximation fails, and the family-wise error rate of Tukey’s procedure is no longer controlled at the nominal level.

**Alternatives**:
- Kruskal-Wallis test followed by Dunn’s test with appropriate correction.
- Robust methods based on trimmed means and Winsorized variances.

### Scenario I: Unplanned Multiple Comparisons
Tukey’s HSD controls the family-wise error rate for *all possible pairwise comparisons*. When a researcher first looks at the data and then decides which comparisons to test (or tests all possible pairs without pre-specification), the actual error rate is no longer controlled, even if Tukey is used.

**Technical consequence**: This is a form of data-dependent analysis that invalidates the frequentist guarantees of the procedure.

**Best practice**:
- Pre-register the comparisons of interest.
- If exploratory, use more conservative corrections or report results as exploratory with appropriate caveats.


## 4. Consequences of Using Inappropriate Methods

- **Invalid p-values and confidence intervals**: The actual coverage probability can be far from the nominal 95%.
- **Inflated family-wise Type I error** in multiple comparison procedures.
- **Misleading effect size estimates** and scientific conclusions.
- **Reproducibility crisis contribution**: Many published findings using classical methods on data that clearly violate assumptions have later failed to replicate.
- **Loss of statistical power** in some cases (robust methods can sometimes be more powerful than classical ones under violation).


## 5. Simulation Illustration

The simulation below demonstrates how quickly performance degrades with small n and skewness.


In [ ]:
import numpy as np
from scipy import stats

np.random.seed(42)
n1 = n2 = 8
mean_diff = 5
skew = 4
n_sim = 2000

results_classic = []
results_welch = []

for _ in range(n_sim):
    g1 = np.random.exponential(scale=10/skew, size=n1) * skew
    g2 = np.random.normal(15 + mean_diff, 8, n2)
    
    p_classic = stats.ttest_ind(g1, g2, equal_var=True)[1]
    p_welch = stats.ttest_ind(g1, g2, equal_var=False)[1]
    
    results_classic.append(p_classic < 0.05)
    results_welch.append(p_welch < 0.05)

print(f"Rejection rate (classic t-test): {np.mean(results_classic):.3f}")
print(f"Rejection rate (Welch t-test):   {np.mean(results_welch):.3f}")
print("Note how performance differs under skewness and heteroscedasticity.")


## 6. Summary of Technical Recommendations

| Situation | Preferred Approach | Key Reason |
|-----------|--------------------|------------|
| Two categorical, sparse table | Fisher’s Exact or exact Monte Carlo | Chi-square approximation fails |
| Paired/repeated categorical | McNemar’s test | Dependence violates independence assumption |
| Small n + skewness/outliers | Yuen’s t-test or bootstrap | Classical mean/variance have low breakdown point |
| Heteroscedasticity + unequal n | Welch ANOVA + Games-Howell | Classical MSE is biased |
| Strong non-normality, moderate n | Robust methods or permutation tests | Better control of error rates |
| Exploratory multiple comparisons | Pre-registration or conservative correction | Data-dependent testing invalidates guarantees |
